In [ ]:
from dotenv import load_dotenv

load_dotenv()

print("✅ Environment variables loaded.")

✅ Environment variables loaded.


In [3]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

In [4]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

In [5]:
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key="blog_outline",  # The result of this agent will be stored in the session state with this key.
)

print("✅ outline_agent created.")

✅ outline_agent created.


In [6]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The `{blog_outline}` placeholder automatically injects the state value from the previous agent's output.
    instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft",  # The result of this agent will be stored with this key.
)

print("✅ writer_agent created.")

✅ writer_agent created.


In [7]:
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This agent receives the `{blog_draft}` from the writer agent's output.
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  # This is the final output of the entire pipeline.
)

print("✅ editor_agent created.")

✅ editor_agent created.


In [8]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


C:\Users\hp\AppData\Local\Temp\ipykernel_4716\3624535915.py:1: DeprecationWarning: SequentialAgent is deprecated and will be removed in future versions. Please use Workflow instead.
  root_agent = SequentialAgent(


In [9]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post about the benefits of multi-agent systems for software developers"
)

OutlineAgent > ## Outline: Unleash Your Development Superpowers: The Magic of Multi-Agent Systems

**Introduction Hook:**

*   Ever feel like you're juggling too many tasks, or that your software projects are becoming increasingly complex to manage? What if there was a way to empower your code to work smarter, not just harder? Enter multi-agent systems (MAS), a powerful paradigm shift that's transforming how we build software.

**Main Sections:**

**1. Breaking Down Complexity: Divide and Conquer with Agents**

*   **Modularization on Steroids:** MAS allow you to decompose complex problems into smaller, independent, and more manageable "agent" components. Each agent can be responsible for a specific task or domain.
*   **Specialized Skills, Better Performance:** Assign specific roles and expertise to agents. This specialization leads to more efficient and focused processing, as each agent excels at its defined function.
*   **Easier Debugging and Maintenance:** Troubleshooting becomes 